Here's a high-level overview of the flow of the code:

1. Import necessary libraries: This includes libraries for handling arrays (NumPy), image processing (OpenCV, skimage), medical image processing (nibabel), machine learning (Keras), and others.

2. Define helper functions: These are functions that will be used throughout the code for various tasks such as plotting images and contours, converting images to RGB, etc.

3. Define paths for the datasets: These are the paths to the directories where the MRI images and masks for Site A and Site B are stored.

4. Define Dataset and Dataloader classes: These classes are used to represent the dataset and to load data from the dataset in batches, respectively.

5. Load the MRI images and masks: This is done using the nibabel library. The images are then preprocessed (e.g., normalized, converted to binary masks, etc.) and low-information slices are removed.

6. Split the dataset into training, validation, and test sets: This is done using the split-folders library.

7. Convert the 3D MRI images to 2D slices and save them: This involves converting the single-channel images to pseudo RGB images, resizing the images, and saving them as 2D slices.

8. Load the data using the Dataset and Dataloader classes: This involves creating instances of the Dataset class for the training, validation, and test sets, and then creating instances of the Dataloader class to load data from these datasets in batches.

9. Train the model: This involves defining the model architecture, compiling the model, defining callbacks, and then training the model using the training and validation data.

10. Evaluate the model: This involves using the trained model to make predictions on the test data and then evaluating the performance of the model.

11. Save the model: This involves saving the trained model so that it can be used later without having to retrain it.

12. Plot the training history: This involves plotting the training and validation loss and accuracy over each epoch to visualize how the model's performance changed during training.


In [ ]:
import os
import numpy as np
import cv2
import nibabel as nib
import keras
import glob
import random
from matplotlib import pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from skimage.transform import resize
import splitfolders

In [ ]:
# Python
def plot_contour(ax, img, mask, slice_index, title):
    binary_mask = (mask[:,:,slice_index] > 0).astype(np.uint8)
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    ax.imshow(img[:,:,slice_index], cmap='gray')
    ax.contour(binary_mask, colors='green', linewidths=1)
    ax.set_title(title)

def plot_single_img_mask_contour(img, mask, slice_index):
    fig, axes = plt.subplots(1, 3, figsize=(10,4))
    titles = ['Image', 'Mask', 'Mask Boundary Overlay']
    for ax, title in zip(axes, titles):
        plot_contour(ax, img, mask, slice_index, title)
    plt.show()

def plot_img_contour(ax, img, mask, slice_index, i):
    titles = [f'Image {i + 1}', f'Mask {i + 1}', f'Mask Overlay {i + 1}']
    for ax, title in zip(ax, titles):
        plot_contour(ax, img, mask, slice_index, title)

def multi_image_view(image_list, mask_list, slice_indices):
    num_images = len(image_list)
    fig, axes = plt.subplots(num_images, 3, figsize=(15, 5 * num_images))
    for i in range(num_images):
        plot_img_contour(axes[i], image_list[i], mask_list[i], slice_indices[i], i)
    plt.tight_layout()
    plt.show()

In [ ]:
# Python
def process_mri_images(site_folder, threshold, main_folder_path, output_folder_path):
    folder_name = site_folder.split('/')[-1]
    files = os.listdir(site_folder)
    mri_files = [f for f in files if f.endswith('.nii.gz') and ('segmentation' not in f.lower())]
    low_info_slices_dict = {}

    for mri_file in mri_files:
        mri_path = os.path.join(site_folder, mri_file)
        segmentation_path = os.path.join(site_folder, f"{mri_file.split('.')[0]}_segmentation.nii.gz")

        mri_img = nib.load(mri_path).get_fdata()
        seg_img = nib.load(segmentation_path).get_fdata().astype(np.uint8)

        # Set segmentation mask to binary
        seg_img[seg_img > 0] = 1

        # Calculate slice information
        slice_information = np.sum(seg_img, axis=(0, 1)) / (seg_img.shape[0] * seg_img.shape[1])

        # Identify slices with less than the specified threshold
        low_info_slices = np.where(slice_information < threshold)[0]

        # Remove the identified low-info slices
        filtered_mri_img = np.delete(mri_img, low_info_slices, axis=2)
        filtered_seg_img = np.delete(seg_img, low_info_slices, axis=2)

        # Save low-info slice indices to dictionary
        low_info_slices_dict[os.path.join(folder_name, mri_file)] = [mri_img.shape, filtered_mri_img.shape, low_info_slices.tolist()]

        # Save the modified 3D image
        np.save(os.path.join(output_folder_path, folder_name, 'mri_images', f"{mri_file.split('.')[0]}.npy"), filtered_mri_img)
        np.save(os.path.join(output_folder_path, folder_name, 'mri_masks', f"{mri_file.split('.')[0]}-seg.npy"), filtered_seg_img)

    return low_info_slices_dict

# Constants
THRESHOLD = 0.01
MAIN_FOLDER_PATH = '/content/drive/MyDrive/Prostate_Data/'
OUTPUT_FOLDER_PATH = '/content/drive/MyDrive/Processed_Prostate_Data/Non_Normalized_Data/'

# Process MRI images
site_folders = [f.path for f in os.scandir(MAIN_FOLDER_PATH) if f.is_dir()]
for site_folder in site_folders:
    low_info_slices_dict = process_mri_images(site_folder, THRESHOLD, MAIN_FOLDER_PATH, OUTPUT_FOLDER_PATH)

In [ ]:
# Python
def split_data(input_folder, output_folder, seed=42, ratio=(0.7, 0.2, 0.1)):
    splitfolders.ratio(input_folder, output=output_folder, seed=seed, ratio=ratio, group_prefix=None)

# Constants
INPUT_BASE_PATH = '/content/drive/MyDrive/Processed_Prostate_Data/Non_Normalized_Data/'
OUTPUT_BASE_PATH = '/content/drive/MyDrive/Processed_Prostate_Data/Non_Normalized_Final_Data/'

# Sites
sites = ['Site-A-RUNMC', 'Site-B-BMC']

# Split data for each site
for site in sites:
    input_folder = os.path.join(INPUT_BASE_PATH, site)
    output_folder = os.path.join(OUTPUT_BASE_PATH, f'{site}-Splited')
    split_data(input_folder, output_folder)

In [ ]:
# Define helper functions
def plot_contour(single_img, seg_img, slice_index):
    binary_mask = (seg_img[:,:,slice_index] > 0).astype(np.uint8)
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    plt.title('Mask Boundary Overlay')
    plt.imshow(single_img[:,:,slice_index], cmap='gray')
    plt.contour(binary_mask, colors='green', linewidths=1)
    plt.show()

def plot_single_img_mask_contour(img, mask, slice_index):
    plt.figure(figsize=(10,4))
    plt.subplot(131)
    plt.imshow(img[:,:,slice_index], cmap='gray')
    plt.title('Image')
    plt.subplot(132)
    plt.imshow(mask[:,:,slice_index])
    plt.title('Mask')
    plt.subplot(133)
    plot_contour(img, mask, slice_index )
    plt.show()

def plot_img_contour(ax, single_img, seg_img, slice_index, i):
    binary_mask = (seg_img[:, :, slice_index] > 0).astype(np.uint8)
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    ax[0].imshow(single_img[:, :, slice_index], cmap='gray')
    ax[0].set_title(f'Image {i + 1}')
    ax[1].imshow(seg_img[:, :, slice_index], cmap='gray')
    ax[1].set_title(f'Mask {i + 1}')
    ax[2].imshow(single_img[:, :, slice_index], cmap='gray')
    ax[2].contour(binary_mask, colors='green', linewidths=1)
    ax[2].set_title(f'Mask Overlay {i + 1}')

def multi_image_view(image_list, mask_list, slice_indices):
    num_images = len(image_list)
    fig, axes = plt.subplots(nrows=num_images, ncols=3, figsize=(15, 5 * num_images))
    for i in range(num_images):
        plot_img_contour(axes[i], image_list[i], mask_list[i], slice_indices[i], i)
    plt.tight_layout()
    plt.show()

def convert_to_rgb(image):
    rgb_image = np.stack((image,) * 3, axis=-1)
    return rgb_image

# Define paths for Site A and Site B datasets
local_path = '/content/drive/MyDrive/Non_Normalized_Resized224/'

A_train_img_dir = local_path +'Site-A-RUNMC-Splited/train/mri_images/'
A_train_mask_dir = local_path +'Site-A-RUNMC-Splited/train/mri_masks/'
A_val_img_dir = local_path + 'Site-A-RUNMC-Splited/val/mri_images/'
A_val_mask_dir = local_path+'Site-A-RUNMC-Splited/val/mri_masks/'
A_test_img_dir = local_path+'Site-A-RUNMC-Splited/test/mri_images/'
A_test_mask_dir = local_path + 'Site-A-RUNMC-Splited/test/mri_masks/'

B_train_img_dir = local_path + 'Site-B-BMC-Splited/train/mri_images/'
B_train_mask_dir = local_path +'Site-B-BMC-Splited/train/mri_masks/'
B_val_img_dir = local_path + 'Site-B-BMC-Splited/val/mri_images/'
B_val_mask_dir = local_path + 'Site-B-BMC-Splited/val/mri_masks/'
B_test_img_dir = local_path + 'Site-B-BMC-Splited/test/mri_images/'
B_test_mask_dir = local_path + 'Site-B-BMC-Splited/test/mri_masks/'

# Define Dataset and Dataloader classes
class Dataset:
    def __init__(self, images_dir, masks_dir, preprocessing=None):
        self.images_fps = [os.path.join(images_dir, image_id) for image_id in sorted(os.listdir(images_dir))]
        self.masks_fps = [os.path.join(masks_dir, mask_id) for mask_id in sorted(os.listdir(masks_dir))]
        self.preprocessing = preprocessing

    def __getitem__(self, idx):
        image = np.load(self.images_fps[idx]).astype(np.float32)
        mask = np.load(self.masks_fps[idx]).astype(np.float32)
        if self.preprocessing:
            image = self.preprocessing(image)
        return image, mask

    def __len__(self):
        return len(self.images_fps)

class Dataloader(keras.utils.Sequence):
    def __init__(self, dataset, batch_size=1, shuffle=False):
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(dataset))
        self.on_epoch_end()

    def __getitem__(self, i):
        start = i * self.batch_size
        stop = (i + 1) * self.batch_size
        data = [self.dataset[j] for j in range(start, stop)]
        batch = [np.stack(samples, axis=0) for samples in zip(*data)]
        return batch

    def __len__(self):
        return len(self.indexes) // self.batch_size

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

In [ ]:
def train_and_evaluate_model(backbone, site, preprocess_input, local_path, epoch, lr, train_img_dir, train_mask_dir, val_img_dir, val_mask_dir, test_img_dir, test_mask_dir):
    # Create model
    model = sm.Unet(backbone, classes=n_classes, activation=activation, encoder_weights='imagenet')
    model.compile(optim, total_loss, metrics)

    # Create callbacks
    callbacks = create_callbacks(local_path = local_path, site= site, backbone= backbone, epoch=epoch, lr=lr)

    # Create datasets
    train_dataset = Dataset(train_img_dir, train_mask_dir, preprocessing=preprocess_input)
    valid_dataset = Dataset(val_img_dir, val_mask_dir, preprocessing=preprocess_input)

    # Initialize dataloaders
    train_dataloader = Dataloader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    valid_dataloader = Dataloader(valid_dataset, batch_size=1, shuffle=False)

    # Train model
    history = model.fit_generator(
        train_dataloader,
        steps_per_epoch=len(train_dataloader),
        epochs=epoch,
        callbacks=callbacks,
        validation_data=valid_dataloader,
        validation_steps=len(valid_dataloader),
    )

    # Save model
    save_model(local_path = local_path, site = site, backbone = backbone , model = model, epoch = epoch, lr=lr)

    # Plot history
    plot_history(history= history)

    # Evaluate model
    test_dataset = Dataset(test_img_dir, test_mask_dir, preprocessing=preprocess_input)
    test_dataloader = Dataloader(test_dataset, batch_size=1, shuffle=False)

    # Load best weights
    model.load_weights(local_path+f'Saved_Models/callbacks/Site_{site}/{backbone}_model_{site}_{epoch}epoch_LR{str(lr).replace(".","")}.h5')

    # Use evaluate instead of deprecated evaluate_generator
    scores = model.evaluate(test_dataloader)

    print("Loss: {:.5}".format(scores[0]))
    for metric, value in zip(metrics, scores[1:]):
        print("mean {}: {:.5}".format(metric.__name__, value))

    return model, history

In [ ]:
model1, history1A = train_and_evaluate_model(BACKBONE1, 'A', preprocess_input1, local_path, epoch, LR, A_train_img_dir, A_train_mask_dir, A_val_img_dir, A_val_mask_dir, A_test_img_dir, A_test_mask_dir)
model1, history1B = train_and_evaluate_model(BACKBONE1, 'B', preprocess_input1, local_path, epoch, LR, B_train_img_dir, B_train_mask_dir, B_val_img_dir, B_val_mask_dir, B_test_img_dir, B_test_mask_dir)
model2, history2A = train_and_evaluate_model(BACKBONE2, 'A', preprocess_input2, local_path, epoch, LR, A_train_img_dir, A_train_mask_dir, A_val_img_dir, A_val_mask_dir, A_test_img_dir, A_test_mask_dir)
model2, history2B = train_and_evaluate_model(BACKBONE2, 'B', preprocess_input2, local_path, epoch, LR, B_train_img_dir, B_train_mask_dir, B_val_img_dir, B_val_mask_dir, B_test_img_dir, B_test_mask_dir)
model3, history3A = train_and_evaluate_model(BACKBONE3, 'A', preprocess_input3, local_path, epoch, LR, A_train_img_dir, A_train_mask_dir, A_val_img_dir, A_val_mask_dir, A_test_img_dir, A_test_mask_dir)
model3, history3B = train_and_evaluate_model(BACKBONE3, 'B', preprocess_input3, local_path, epoch, LR, B_train_img_dir, B_train_mask_dir, B_val_img_dir, B_val_mask_dir, B_test_img_dir, B_test_mask_dir)

In [ ]:
import keras
import segmentation_models as sm

# Define common parameters
n_classes = 1
activation = 'sigmoid'
epoch = 34
LR = 0.0001
optim = keras.optimizers.Adam(LR)
BATCH_SIZE = 10

# Define loss and metrics
dice_loss = sm.losses.DiceLoss()
focal_loss = sm.losses.BinaryFocalLoss()
total_loss = dice_loss + (1 * focal_loss)
metrics = [sm.metrics.IOUScore(threshold=0.5), sm.metrics.FScore(threshold=0.5)]

# Define backbones and their preprocessing functions
backbones = ['vgg19', 'resnet50', 'efficientnetb0']
preprocess_inputs = [sm.get_preprocessing(backbone) for backbone in backbones]

# Define paths for Site A and Site B datasets
A_train_img_dir = ...
A_train_mask_dir = ...
A_val_img_dir = ...
A_val_mask_dir = ...
A_test_img_dir = ...
A_test_mask_dir = ...

B_train_img_dir = ...
B_train_mask_dir = ...
B_val_img_dir = ...
B_val_mask_dir = ...
B_test_img_dir = ...
B_test_mask_dir = ...

# Define local path for saving models and logs
local_path = ...

# Define function for training and evaluating model
def train_and_evaluate_model(backbone, site, preprocess_input, local_path, epoch, lr, train_img_dir, train_mask_dir, val_img_dir, val_mask_dir, test_img_dir, test_mask_dir):
    # Create model
    model = sm.Unet(backbone, classes=n_classes, activation=activation, encoder_weights='imagenet')
    model.compile(optim, total_loss, metrics)

    # Create callbacks
    callbacks = create_callbacks(local_path = local_path, site= site, backbone= backbone, epoch=epoch, lr=lr)

    # Create datasets
    train_dataset = Dataset(train_img_dir, train_mask_dir, preprocessing=preprocess_input)
    valid_dataset = Dataset(val_img_dir, val_mask_dir, preprocessing=preprocess_input)

    # Initialize dataloaders
    train_dataloader = Dataloader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    valid_dataloader = Dataloader(valid_dataset, batch_size=1, shuffle=False)

    # Train model
    history = model.fit_generator(
        train_dataloader,
        steps_per_epoch=len(train_dataloader),
        epochs=epoch,
        callbacks=callbacks,
        validation_data=valid_dataloader,
        validation_steps=len(valid_dataloader),
    )

    # Save model
    save_model(local_path = local_path, site = site, backbone = backbone , model = model, epoch = epoch, lr=lr)

    # Plot history
    plot_history(history= history)

    # Evaluate model
    test_dataset = Dataset(test_img_dir, test_mask_dir, preprocessing=preprocess_input)
    test_dataloader = Dataloader(test_dataset, batch_size=1, shuffle=False)

    # Load best weights
    model.load_weights(local_path+f'Saved_Models/callbacks/Site_{site}/{backbone}_model_{site}_{epoch}epoch_LR{str(lr).replace(".","")}.h5')

    # Use evaluate instead of deprecated evaluate_generator
    scores = model.evaluate(test_dataloader)

    print("Loss: {:.5}".format(scores[0]))
    for metric, value in zip(metrics, scores[1:]):
        print("mean {}: {:.5}".format(metric.__name__, value))

    return model, history

# Train and evaluate models for each backbone and site
for backbone, preprocess_input in zip(backbones, preprocess_inputs):
    model, history = train_and_evaluate_model(backbone, 'A', preprocess_input, local_path, epoch, LR, A_train_img_dir, A_train_mask_dir, A_val_img_dir, A_val_mask_dir, A_test_img_dir, A_test_mask_dir)
    model, history = train_and_evaluate_model(backbone, 'B', preprocess_input, local_path, epoch, LR, B_train_img_dir, B_train_mask_dir, B_val_img_dir, B_val_mask_dir, B_test_img_dir, B_test_mask_dir)